# SPPM Signed Integer Tests

Focused experiments for the signed partial-power modulus integer class `SPPM`, which builds on `PPM`.

Automated verification: `tests/test_06_sppm.py`. The numbered pytest files establish
the prerequisite hierarchy before this class or module.

In [1]:
from pathlib import Path
import sys


def find_repo_root(start=None):
    path = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (path, *path.parents):
        if (candidate / "rns_pypal").exists() and (candidate / "tests").exists():
            return candidate
    raise RuntimeError("Could not find the RNS-PYPAL repository root")


repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from rns_pypal import (
    NEGATIVE,
    POSITIVE,
    PPM,
    RNSNumberSystem,
    SIGN_INVALID,
    SIGN_VALID,
    SPPM,
)

repo_root

WindowsPath('D:/Projects/RNS-APAL-REPO/RNS-APAL')

## Signed range and representation

In [2]:
system = RNSNumberSystem(
    moduli=[2, 3],
    powers=[4, 1],
    name="signed-48",
)

print("unsigned dynamic range:", system.dynamic_range)
print("signed usable range: -24..23")
print("PPM usable range:", PPM.format_usable_range(system))

unsigned dynamic range: 48
signed usable range: -24..23
PPM usable range: 0..47


In [3]:
x = SPPM(-1, system=system)

print("native residues:")
x.print_whdr()

print("signed value:", x.format_value())
print("sign flag:", x.sign_flag)
print("sign valid:", x.sign_valid)
print("calc sign:", x.calc_sign())

native residues:
16 3
-- -
15 2
signed value: -1
sign flag: 1
sign valid: 1
calc sign: 1


In [4]:
a = SPPM("-15", system=system)
b = SPPM("-0xf", system=system)

print(a.format_value(), a.format_native())
print(b.format_value(), b.format_native())
print("same residues:", a.format_native() == b.format_native())

-15 1 0
-15 1 0
same residues: True


## Sign state and validation

In [5]:
x = SPPM(-7, system=system)

x.sign_flag = POSITIVE
x.sign_valid = SIGN_INVALID

print("before:")
print("value:", x.format_value())
print("sign flag:", x.sign_flag)
print("sign valid:", x.sign_valid)

x.calc_set_sign()

print("\nafter calc_set_sign:")
print("value:", x.format_value())
print("sign flag:", x.sign_flag)
print("sign valid:", x.sign_valid)

before:
value: -7
sign flag: 0
sign valid: 0

after calc_set_sign:
value: -7
sign flag: 1
sign valid: 1


In [6]:
x = SPPM(-3, system=system)

print("true value:", x.format_value())
print("mismatch before:", x.sign_mismatch())

x.sign_flag = POSITIVE
x.sign_valid = SIGN_VALID

print("mismatch after corrupting sign flag:", x.sign_mismatch())
print("residue-ground-truth value still:", x.format_value())

true value: -3
mismatch before: False
mismatch after corrupting sign flag: True
residue-ground-truth value still: -3


## Signed arithmetic

In [7]:
x = SPPM(4, system=system)
y = SPPM(5, system=system)

x.add(y)

print("4 + 5 =", x.format_value())
print("sign flag:", x.sign_flag)
print("sign valid:", x.sign_valid)

4 + 5 = 9
sign flag: 0
sign valid: 1


In [8]:
x = SPPM(-4, system=system)
y = SPPM(-5, system=system)

x.add(y)

print("-4 + -5 =", x.format_value())
print("sign flag:", x.sign_flag)
print("sign valid:", x.sign_valid)

-4 + -5 = -9
sign flag: 1
sign valid: 1


In [9]:
x = SPPM(5, system=system)

x.sub(SPPM(-3, system=system))
print("5 - -3 =", x.format_value())
print("sign:", x.sign_flag, "valid:", x.sign_valid)

x.sub(SPPM(10, system=system))
print("8 - 10 =", x.format_value())
print("sign:", x.sign_flag, "valid:", x.sign_valid)

x.calc_set_sign()
print("after sign recovery:", x.sign_flag, x.sign_valid)

5 - -3 = 8
sign: 0 valid: 1
8 - 10 = -2
sign: 0 valid: 0
after sign recovery: 1 1


In [10]:
x = SPPM(-4, system=system)
y = SPPM(5, system=system)

x.mult(y)

print("-4 * 5 =", x.format_value())
print("sign flag:", x.sign_flag)
print("sign valid:", x.sign_valid)

-4 * 5 = -20
sign flag: 1
sign valid: 1


In [11]:
x = SPPM(7, system=system)

x.negate()
print("negated:", x.format_value())
x.print_whdr()

x.abs()
print("absolute:", x.format_value())
x.print_whdr()

negated: -7
16 3
-- -
9  2
absolute: 7
16 3
-- -
7  1


## Comparison and boundaries

In [12]:
cases = [
    (2, -1),
    (-1, 2),
    (-1, -2),
    (5, 5),
]

for left, right in cases:
    a = SPPM(left, system=system)
    b = SPPM(right, system=system)

    print(f"{left} > {right} ?", bool(a.compare(b)))

2 > -1 ? True
-1 > 2 ? False
-1 > -2 ? True
5 > 5 ? False


In [13]:
for n in [-24, -23, -1, 0, 1, 22, 23]:
    x = SPPM(n, system=system)
    print(n, "=>", x.format_native(), "=>", x.format_value(), "sign:", x.sign_flag)

-24 => 8 0 => -24 sign: 1
-23 => 9 1 => -23 sign: 1
-1 => 15 2 => -1 sign: 1
0 => 0 0 => 0 sign: 0
1 => 1 1 => 1 sign: 0
22 => 6 1 => 22 sign: 0
23 => 7 2 => 23 sign: 0
